## Importing libraries

In [ ]:
import numpy as np
import pandas as pd
import pickle
import re
from functions.functions import *

## Importing data

Columns to drop

In [ ]:
columns_to_not_use = [
# Same values for all rows
    'record_type' ,
    'record_subclass' ,
    'confirmation_location' ,
    'dia_bp' ,
    'group_number' ,
    'manufacturer_code' ,
    'patient_age_units' ,
    'patient_height' ,
    'patient_weight' ,
    'reason_for_procedure' ,
    'sys_bp' ,
    'text_size' ,
    'pp_interval' ,

# No value at all
    'assisting_provider' ,
    'attending_phy' ,
    'patient_middle_name' ,
    'patient_secondary_id' ,
    'patient_ssn' ,
    'procedural_diagnosis' ,
    'referral_source' ,
    'referring_phy' ,


# Probably irrelevant
    'reason_for_change' ,

# "Personal" data - with many NA
    'patient_race_str' ,

# "Personal" data - useless without values
    'patient_weight_units' ,
    'patient_height_units' ,
]

Columns to format in a particular type

In [ ]:
sure_string_cols = [
    'pseudopaz',
    'patient_date_of_birth', # Because of the inconsistency ''dd.mm.yyyy' vs 'mm/dd/yyyy'
]
dtype = {col: 'string' for col in sure_string_cols}

string_cat_cols = [
    'site_number', # ???
    'record_status',
    'medications',
    'patient_height_units' ,
    'patient_location' ,
    'patient_race' ,
    'patient_race_str' ,
    'patient_room' ,
    'patient_sex' ,
    'patient_weight_units' ,
    'transmit_device_number' ,
    'reason_for_change' ,
]
dtype.update({col: 'string' for col in string_cat_cols})

Import the excel file

In [ ]:
df = pd.read_excel('../raw_data-originals/dati_escribe.xlsx',
    usecols=lambda col: col not in columns_to_not_use,
    dtype_backend='numpy_nullable', # For allowing the `string` dtype
    dtype=dtype
)

# Make the index start from 2 (as the original excel)
df.index = df.index + 2

## Parsing and reformatting

Defining `format_phy_name` function

In [ ]:
def format_phy_name(value):
    '''
    Requires pandas.
    TODO
    '''
    # Handle NA values
    if pd.isna(value):
        return pd.NA

    # Count whitespaces
    dot_count = value.count('.')

    # Handle the only name with one dot
    if dot_count == 1:
        return value

    elif dot_count == 2:
        second_dot = value.rfind('.')
        name = value[second_dot -1 : second_dot]
        surname = value[value.rfind('.') +1 : ].strip()
        formatted_name = 'DR-' + name + '_' + surname
        
        return formatted_name

### Formatting `overreading_phy` column

In [ ]:
# Only name with one dot
df['overreading_phy'] = df['overreading_phy'].replace('DR. GP BROSO', 'DR-GP_BROSO')

# Run the function
df['overreading_phy'] = df['overreading_phy'].apply(format_phy_name).astype('string')

### Splitting and formatting `interp_text` column

In [ ]:
def split_column(value):

    # If row is already a NA value, leave unchanged
    if pd.isna(value):
        return pd.Series([pd.NA, pd.NA, pd.NA])

    # Split on the first ~~ from right
    parts = value.rsplit('~~', maxsplit=1)

    # Place a NA in the missing splitted parts
    for i, part in enumerate(parts):
        if part == '':
            parts[i] = pd.NA

    # Parse 'parts[1]'
    if parts[1].startswith('REPORT NON CONFERM'):
        doc, datetime = 'RNC', pd.NaT
    elif parts[1].startswith('Refertato da _________________'):
        doc, datetime = 'phy_NA', pd.NaT
    elif parts[1].startswith('Refertato da '):
        doc_pro, date, hour = (re.sub(' +', ' ', parts[1])).strip().rsplit(' ', maxsplit=2)
        doc = format_phy_name(doc_pro[13:])
        datetime = pd.to_datetime('{} {}'.format(date, hour), format='%d/%m/%Y %H.%M.%S')

    if len(parts) == 2:
        return pd.Series([parts[0], doc, datetime])
    else:
        raise Exception('Abnormal row')

Pre-format row 1685, the only problematic

In [ ]:
original = df.at[1685, 'interp_text']
df.at[1685, 'interp_text'] = '~' + original

In [ ]:
# Get position and split
original_col_name = 'interp_text'
original_position = df.columns.get_loc(original_col_name)

# Split the column
split_df = df[original_col_name].apply(split_column)
split_df.columns = ['diagnosis', 'phy', 'phy_date']
split_df[['diagnosis', 'phy']] = split_df[['diagnosis', 'phy']].astype('string')

# Reconstruct dataframe: columns before + new columns + columns after
df = pd.concat([
    df.iloc[:, :original_position
        # +1 # Decomment for keeping the original column
    ],
    split_df,
    df.iloc[:, original_position+1:]
], axis=1)

### Merging similar values in columns

#### `comment_field`

Values are `{..., 'Dolore toracico': 8,'Dolore torace': 2, ...}` - `Dolore torace` values are merged to `Dolore toracico`

In [ ]:
df['comment_field'] = df['comment_field'].replace('Dolore torace', 'Dolore toracico')

#### `note1`

Merging:
- `DOL. TORACICO` and `DOL TOR` and `DOLORETORAC` and `DOLORE TORACICO`
    - to `TORACALGIA`
- `ADDOMINALG` and `DOLORE ADDOMIN`
    - to `ADDOMINALGIA`
- `Dispnea`
    - to `DISPNEA`

In [ ]:
df['note1'] = df['note1'].replace(['DOLORE TORACICO', 'DOLORETORAC', 'DOL. TORACICO', 'DOL TOR'], 'TORACALGIA')
df['note1'] = df['note1'].replace(['ADDOMINALG', 'DOLORE ADDOMIN'], 'ADDOMINALGIA')
df['note1'] = df['note1'].replace(['Dispnea'], 'DISPNEA')

#### `note2` (tanti auguri)

Merging:
- `.` and `.`
    - to `.`

In [ ]:
# df[''] = df[''].replace([''], '')

#### `patient_location`

Merging:
- `OTO`
    - to `Oto`

In [ ]:
df['patient_location'] = df['patient_location'].replace(['OTO'], 'Oto')

#### ``

Merging:
- `.` and `.`
    - to `.`

In [ ]:
# df[''] = df[''].replace([''], '')

### Correcting and reformatting `patient_date_of_birth` column

#### Proof of consistency between american and european formatting

Most of the dates are written with `.` as separator, only 33 of them with `/`:

In [ ]:
dot_rows = df['patient_date_of_birth'].str.contains(r'\.', na=False).sum()
slash_rows = df['patient_date_of_birth'].str.contains('/', na=False).sum()
print('Number of rows formatted with dot:', dot_rows)
print('Number of rows formatted with slash:', slash_rows)

Hypotesis: _dates written with dots as separator are written in the european way (`dd.mm.yyyy`), dates with slash are written in the american way (`mm/dd/yyyy`)_.

There can't be an absolute proof of this, because some day numbers are lower or equal than 12.

But:
* if, for slash dates, the first field never exceeds 12 but the second often does
* if, for dot dates, the first field often exceeds 12 while the second never does

the probability that the hypotesis is true is very high. So let's test it.

In [ ]:
# Remove surrounding spaces
df['patient_date_of_birth'] = df['patient_date_of_birth'].str.strip()

# Separate rows by separator
slash_mask = df['patient_date_of_birth'].str.contains('/', na=False)
dot_mask = df['patient_date_of_birth'].str.contains(r'\.', na=False)

# Filter values from patient_date_of_birth column only for rows where slash_mask is True.
slash_dates = df.loc[slash_mask, 'patient_date_of_birth']
dot_dates = df.loc[dot_mask, 'patient_date_of_birth']

# Split the dates components and rename the columns for clarity
slash_parts = slash_dates.str.split('/', expand=True).astype(int)
slash_parts.columns = ['s_part1', 's_part2', 's_year']
dot_parts = dot_dates.str.split(r'\.', expand=True).astype(int)
dot_parts.columns = ['d_part1', 'd_part2', 'd_year']

Proof for slash-formatted days

In [ ]:
slash_first_gt12 = (slash_parts['s_part1'] > 12).sum()
slash_second_gt12 = (slash_parts['s_part2'] > 12).sum()

print('SLASH DATES')
print(f'Total slash dates: {len(slash_parts)}')
print(f'First field > 12 : {slash_first_gt12}')
print(f'Second field > 12: {slash_second_gt12}')

# Rows that prove mm/dd/yyyy
slash_proof = slash_dates[slash_parts['s_part2'] > 12]
slash_proof.shape

Proof for dot-formatted days

In [ ]:
dot_first_gt12 = (dot_parts['d_part1'] > 12).sum()
dot_second_gt12 = (dot_parts['d_part2'] > 12).sum()

print('DOT DATES')
print(f'Total dot dates: {len(dot_parts)}')
print(f'First field > 12 : {dot_first_gt12}')
print(f'Second field > 12: {dot_second_gt12}')

# Rows that prove dd.mm.yyyy
dot_proof = dot_dates[dot_parts['d_part1'] > 12]

So in a total of 1787 dates:
* 16 dates out of 33 suggest that slash-formatted are written in the format `mm/dd/yyyy`
* 1001 dates out of 1754 suggest that dot-formatted are written in the format `mm/dd/yyyy`
* no date goes against the hypotesis

#### Reformatting and converting to datetime type

In [ ]:
# Clean eventual whitespaces
df['patient_date_of_birth'] = df['patient_date_of_birth'].str.strip()

# Make masks for the two separators
slash_mask = df['patient_date_of_birth'].str.contains('/', na=False)
dot_mask = df['patient_date_of_birth'].str.contains(r'\.', na=False)

# Temporary Series to hold parsed dates
parsed_dates = pd.Series(pd.NaT, index=df.index)

# Parse slash dates (mm/dd/yyyy)
parsed_dates.loc[slash_mask] = pd.to_datetime(
    df.loc[slash_mask, 'patient_date_of_birth'],
    format='%m/%d/%Y',
    errors='coerce'
)

# Parse dot dates (dd.mm.yyyy)
parsed_dates.loc[dot_mask] = pd.to_datetime(
    df.loc[dot_mask, 'patient_date_of_birth'],
    format='%d.%m.%Y',
    errors='coerce'
)

# Replace original column
df['patient_date_of_birth'] = parsed_dates

# Check if some rows have failed the parsing
failed_rows = df[df['patient_date_of_birth'].isna()]
print(f'Failed rows: {len(failed_rows)}')

# # Rename column to a different name
# df = df.rename(columns={
#     'patient_date_of_birth': 'normalized_date'
# })

#### Birth dates before 1900

There are some birth dates prior to 1900:

In [ ]:
mask_1900 = df['patient_date_of_birth'] < pd.Timestamp('1900-01-01')

print(f'number of birth dates prior to 1900: {mask_1900.sum()}')
df.loc[mask_1900, 'patient_date_of_birth']# + pd.DateOffset(years=100)

There's a high chance that these are errors, and the true year is `year+100`. After performing this change, the number of inconsistencies between pseudopaz and birthdates drop from 13 to 6.

To run this correction:

In [ ]:
df.loc[mask_1900, 'patient_date_of_birth'] = df.loc[mask_1900, 'patient_date_of_birth'] + pd.DateOffset(years=100)

#### Last inconsistent dates

List pseudopaz with the two inconsistent dates found

In [ ]:
# Count distinct date_of_birth per pseudopaz
dup_counts = df.groupby('pseudopaz')['patient_date_of_birth'].nunique()

# Keep only rows with > 1 distinct DOB
problem_ids = dup_counts[dup_counts > 1]
problem_ids

# filter original df
conflicts = (
    # Filters the dataframe to keep only rows where pseudopaz is in the list of problematic IDs.
    df[df['pseudopaz'].isin(problem_ids.index)]
    # Groups the filtered df by pseudopaz and considers only the DOB column within each group
    .groupby('pseudopaz')['patient_date_of_birth']
    # For each group, it returns distinct DOB values
    .unique()
    # Converts the grouped result back into a normal dataframe (otherwise would be a Series)
    .reset_index()
)

conflicts

In [ ]:
# Pretty printing

for _, row in conflicts.iterrows():
    pseudopaz = row['pseudopaz']
    dobs = row['patient_date_of_birth']
    
    print(f"DOBs of {pseudopaz}:")
    
    for dob in dobs:
        # If it's a datetime, remove the time part
        print(f'   {dob.date() if hasattr(dob, 'date') else dob}')

Match pseudopaz with the two inconsistent dates found and their frequency

In [ ]:
# Count distinct date_of_birth per pseudopaz
dup_counts = df.groupby('pseudopaz')['patient_date_of_birth'].nunique()

# Keep only rows with > 1 distinct DOB
problem_ids = dup_counts[dup_counts > 1]

# Filter original df to problem IDs only
filtered_df = df[df['pseudopaz'].isin(problem_ids.index)]

# Create result list
result_data = []
for problem_id in problem_ids.index:
    # Get all DOBs for this pseudopaz
    dobs = filtered_df[filtered_df['pseudopaz'] == problem_id]['patient_date_of_birth']
    
    # Count occurrences and format dates
    dob_dict = {}
    for dob in dobs:
        formatted_date = dob.strftime('%d-%m-%Y')
        dob_dict[formatted_date] = dob_dict.setdefault(formatted_date, 0) + 1

    result_data.append({'pseudopaz': problem_id, 'DoB:counts': dob_dict})

# Create final dataframe
conflicts = pd.DataFrame(result_data)
conflicts

For each problematic pseudopaz, set as date of birth the most frequent one

In [ ]:
# Iter over the rows of the previoud df
for row in conflicts.itertuples(index=False):
    pseudopaz, dobs = row[0], row[1]

    # Among all keys in dobs, return the key with the highest associated value
    more_frequent = max(dobs, key=dobs.get)

    # Set as the patient_date_of_birth of the current pseudopaz the date more_frequent
    df.loc[df['pseudopaz'] == pseudopaz, 'patient_date_of_birth'] = pd.Timestamp(more_frequent)

### Reformat ECG columns to handle undetected NA data

Proof

In [ ]:
affected_columns = [
    'p_onset' ,
    'p_offset' ,
    'pr_interval' ,
    'p_axis' ,
]

In [ ]:
def mode_dfcolumn(df, column):
    original = df[column].value_counts(dropna=False).to_dict()

    max_key = max(original, key=original.get)

    other_sum = 0
    for key, value in original.items():
        if key != max_key:
            other_sum += value

    result = {
        max_key: original[max_key],
        'other_values': other_sum
    }

    return result

Columns with `0` value outliers

In [ ]:
affected_stats = {}
for column in affected_columns:
    
    affected_stats[column] = mode_dfcolumn(df, column)

affected_stats

In [ ]:
# TODO
# Print stats about affected columns (proportions, percentages, ...)
# and make a little summary plot containing pie plots of the percentages

Reformat to `NA` values

In [ ]:
df[affected_columns[0:-1]] = (df[affected_columns[0:-1]].replace(
    np.int64(0), pd.NA
))

In [ ]:
df[affected_columns[-1]] = (df[affected_columns[-1]].replace(
    np.int64(999), pd.NA
))

### Reorder columns

In [ ]:
df = df[[
    'pseudopaz' ,

# "Personal" data
    'patient_date_of_birth' ,
    'patient_age' ,
    'patient_sex' ,
    'patient_race' ,

# Data about date/hour
    'acq_time' ,
    'transmission_time' ,

# Data about date/hour - with many NaT
    'confirm_time' ,
    'phy_date' ,


# Location
    'site_number' ,

# Location - with many NA
    'patient_location' ,
    'patient_room' ,    


# Diagnosis and medical notes
    'medications' ,
    'comment_field' ,
    'diagnosis' , # Has NAs

# Diagnosis and medical notes - with many NA
    'note1' ,
    'note2' ,


# ECG data (reordered)
    # Temporal (wave timing) features - measured in ms
    'p_onset' , # NA values found
    'p_offset' , # NA values found
    'qrs_onset' ,
    'qrs_offset' ,
    't_offset' ,
    # Interval/duration features - measured in ms
    'pr_interval' , # NA values found
    'qrs_duration' ,
    'qt_interval' ,
    'qtc_interval' ,
    'rr_interval' ,
    # Axis (vector direction) features - measured in deg
    'p_axis' , # NA values found
    'qrs_axis' ,
    't_axis' ,
    # Heart rate feature - measured in bpm
    'v_rate' ,


# Doctor/phy - with many NA
    'overreading_phy' ,
    'phy' ,


# Other data
    'transmit_device_number' ,


# Unknown data
    'record_status' ,
    'record_size' ,
    'sequence_number' ,
]]

### Rename columns

In [ ]:
# df = df.rename(columns={
#     'patient_date_of_birth' : 'date_of_birth',
#     'patient_age' : 'age',
#     'patient_sex' : 'sex',
#     'patient_race' : 'race',
# })

### Drop columns with 50% or more NAs

In [ ]:
# df = drop_columns_by_na_threshold(df,
#     threshold=0.5,
#     return_names=True
# )

drop_columns_by_na_threshold(df, return_names=True,
    threshold=0.5,
)

### Reorder by pseudopaz (and eventually reset index)

In [ ]:
df = df.sort_values(by=['pseudopaz'])

In [ ]:
# df = df.reset_index(drop=True)

## Final checks

In [ ]:
df

In [ ]:
df.info()

## Export

In [ ]:
# # Parquet
# df.to_parquet('../raw_data/parquets/1-escribe.parquet')

# # Pickle
# df.to_pickle('../raw_data/pickles/1-escribe.pickle')
# df['pseudopaz'].to_pickle('../raw_data/pickles/pps/1-escribe-pp.pickle')
# df['pseudopaz'].drop_duplicates().to_pickle('../raw_data/pickles/pps/1-escribe-pp-uniq.pickle')
# with open('../raw_data/pickles/pps/1-escribe-pp-uniq-list.pickle', 'wb') as file:
#     pickle.dump(df['pseudopaz'].drop_duplicates().tolist(), file, protocol=pickle.HIGHEST_PROTOCOL)

# # Excel
# df.to_excel('../raw_data/excel_formatted/1-escribe.xlsx', sheet_name='1-escribe', index=False)

## Further analysis

### ECG checks

#### Basic physiological ranges

In [ ]:
# Example ranges in milliseconds (ms) unless noted

valid_ranges = {
    "pr_interval": (80, 220),
    "qrs_duration": (60, 140),
    "qt_interval": (300, 500),
    "qtc_interval": (350, 550),
    "rr_interval": (300, 2000),  # corresponds ~30–200 bpm
    "v_rate": (30, 220),         # bpm
    "p_axis": (-90, 90),
    "qrs_axis": (-180, 180),
    "t_axis": (-90, 90),
}

for col, (low, high) in valid_ranges.items():
    mask = df[col].notna()
    # Checks how many rows have a value higher/lower than the theoretical higher/lower
    invalid = mask & ((df[col] < low) | (df[col] > high))
    print(col, invalid.sum())

#### Interval consistency checks

In [ ]:
invalid_order = (
    (df["p_onset"] > df["p_offset"]) |
    (df["p_offset"] > df["qrs_onset"]) |
    (df["qrs_onset"] > df["qrs_offset"]) |
    (df["qrs_offset"] > df["t_offset"])
)
invalid_order = invalid_order.fillna(False)
invalid_order.value_counts(dropna=False).to_dict()

In [ ]:
# PR interval ≈ qrs_onset - p_onset
pr_calc = df["qrs_onset"] - df["p_onset"]
pr_mismatch = (pr_calc - df["pr_interval"]).abs() > 5

# QRS duration
qrs_calc = df["qrs_offset"] - df["qrs_onset"]
qrs_mismatch = (qrs_calc - df["qrs_duration"]).abs() > 5

# QT interval
qt_calc = df["t_offset"] - df["qrs_onset"]
qt_mismatch = (qt_calc - df["qt_interval"]).abs() > 10

pr_mismatch = pr_mismatch.fillna(False)
qrs_mismatch = qrs_mismatch.fillna(False)
qt_mismatch = qt_mismatch.fillna(False)

qt_mismatch.value_counts(dropna=False).to_dict()

#### Heart rate consistency

In [ ]:
# HR ≈ 60,000 / RR (ms)
hr_calc = 60000 / df["rr_interval"]
hr_mismatch = (hr_calc - df["v_rate"]).abs() > 10
hr_mismatch = hr_mismatch.fillna(False)

hr_mismatch.value_counts(dropna=False).to_dict()

#### QT vs QTc consistency (possible errors!!)

In [ ]:
# QTc = QT / sqrt(RR in seconds)
qt_sec = df["qt_interval"] / 1000
rr_sec = df["rr_interval"] / 1000

qtc_calc = qt_sec / (rr_sec ** 0.5) * 1000

qtc_mismatch = (qtc_calc - df["qtc_interval"]).abs() > 40
qtc_mismatch = qtc_mismatch.fillna(False)

qtc_mismatch.value_counts(dropna=False).to_dict()

#### Negative or zero durations

In [ ]:
bad_duration = (
    (df["pr_interval"] <= 0) |
    (df["qrs_duration"] <= 0) |
    (df["qt_interval"] <= 0) |
    (df["rr_interval"] <= 0)
)
bad_duration = bad_duration.fillna(False)
bad_duration.value_counts(dropna=False).to_dict()

In [ ]:
# Example: extreme axis combinations
extreme_axis = (
    (df["qrs_axis"].abs() > 150) &
    (df["p_axis"].abs() < 10)
)
extreme_axis = extreme_axis.fillna(False)
extreme_axis.value_counts(dropna=False).to_dict()

### Time span between time columns

In [ ]:
wdf = df.copy()

In [ ]:
# wdf['acq_time'] = wdf['acq_time'].dt.normalize()
# wdf['transmission_time'] = wdf['transmission_time'].dt.normalize()
# wdf['confirm_time'] = wdf['confirm_time'].dt.normalize()
# wdf['phy_date'] = wdf['phy_date'].dt.normalize()

time_cols = [
    'acq_time',
    'transmission_time',
    'confirm_time',
    'phy_date',
]

# Compute row-wise min and max, skipping NaT automatically
row_min = wdf[time_cols].min(axis=1)
row_max = wdf[time_cols].max(axis=1)

# Difference
delta = row_max - row_min

# Boolean mask: True if span > 1 day
mask = delta > pd.Timedelta(days=1) # Set the timedelta!!

# Filter rows
result = wdf.loc[mask, ['pseudopaz'] + time_cols].copy()
result['delta'] = delta[mask]
result = result[['pseudopaz', 'delta'] + time_cols]

print(f'Affected rows: {len(result)}')

result = result.sort_values(by=['delta'])
result

### Comparison between given and computed age

In [ ]:
import datetime
from dateutil.relativedelta import *
import calendar

In [ ]:
def validate_patient_ages(df, ref_time_column, *, within_days=None):
    """
    Validate patient ages by computing age from date of birth to acquisition time
    
    Parameters:
    - df : the df to check against (will not be modified)
    - ref_time_column: the column of the df to use as reference to compute the age
    - within_days (optional):
        * 0: show only rows where ages don't match exactly
        * n>0: show only rows where ages don't match within the specified number of days
    
    Returns a pandas DataFrame that shows the difference between given and computed age
    """
    # Create a copy to avoid modifying original
    df_work = df.copy()
    
    # Convert to datetime
    df_work['patient_date_of_birth'] = pd.to_datetime(df_work['patient_date_of_birth'])
    df_work[ref_time_column] = pd.to_datetime(df_work[ref_time_column])
    
    # Compute age at acquisition time
    computed_ages = []
    for idx in df_work.index:
        dob = df_work.loc[idx, 'patient_date_of_birth'].to_pydatetime()
        acq = df_work.loc[idx, ref_time_column].to_pydatetime()
        age_delta = relativedelta(acq, dob)
        computed_age = age_delta.years
        computed_ages.append(computed_age)
    
    # Create output dataframe
    result_df = pd.DataFrame()
    result_df['pseudopaz'] = df_work['pseudopaz']
    result_df['patient_age'] = df_work['patient_age']
    result_df['computed_age'] = computed_ages
    result_df['patient_date_of_birth'] = df_work['patient_date_of_birth']
    result_df[ref_time_column] = df_work[ref_time_column].dt.date  # Normalize to date only
    
    # Apply filter
    if within_days is None:
        return result_df
    
    elif within_days == 0:
        # Show only rows where ages don't match exactly
        exact_mask = result_df['patient_age'] != result_df['computed_age']
        result_df = result_df[exact_mask]
    
    elif within_days > 0:
        # Check with 15-day margin
        n_days_mask = []
        for idx in df_work.index:
            dob = df_work.loc[idx, 'patient_date_of_birth'].to_pydatetime()
            acq = df_work.loc[idx, ref_time_column].to_pydatetime()
            patient_age = df_work.loc[idx, 'patient_age']
            
            # Calculate date range for the patient_age with ±15 days margin
            age_low = dob + relativedelta(years=patient_age, days=-within_days)
            age_high = dob + relativedelta(years=patient_age + 1, days=within_days)
            
            # Check if acq_time falls within the range
            if not (age_low <= acq < age_high):
                n_days_mask.append(True)
            else:
                n_days_mask.append(False)
        
        result_df = result_df[n_days_mask]
    
    return result_df

Validation of `relativedelta()` and `validate_patient_ages()` functions

In [ ]:
date1 = pd.Timestamp('1949-03-15 00:00:00').to_pydatetime()
date2 = pd.Timestamp('1956-06-25 00:00:00').to_pydatetime()

relativedelta(date2, date1)
# should be 7 years 3 months 10 days

In [ ]:
test_data = {"pseudopaz": ['p_0', 'p_1', 'p_2'],
    "patient_age": [1, 1, 2],
    "patient_date_of_birth": ['2000-01-01', '2000-01-01', '2000-01-01'],
    "acq_time": ['2000-12-31', '2001-01-01', '2001-01-02']
}
test_df = pd.DataFrame(data=test_data)
test_df

validate_patient_ages(test_df, 'acq_time',
    within_days=0
)

Definition of the time column to use as reference to compute the age

In [ ]:
ref_time_column = 'acq_time'

Show only exact mismatches

In [ ]:
result_exact = validate_patient_ages(df, ref_time_column, within_days=175)
result_exact = validate_patient_ages(df, ref_time_column, within_days=0)
result_exact

**RESULTS**

If the mismatched days of births of the 6 pseudopazs:
* are NOT modified, there is no inconsistency between computed and provided age
* are modified, there are 42 exact-day mismatches; the number days after which there are no mismatches is175

Show all rows

In [ ]:
result_all = validate_patient_ages(df, ref_time_column)

## Inconsistency between `pseudopaz` and other columns

#### Proof of the inconsistency

`patient_age`, `patient_date_of_birth`, `patient_race`, `patient_sex` are inconsistent

In [ ]:
# print((df.groupby('pseudopaz')['patient_age'].nunique() == 1).all())
# print((df.groupby('pseudopaz')['patient_date_of_birth'].nunique() == 1).all())
# print((df.groupby('pseudopaz')['patient_race'].nunique() == 1).all())
# print((df.groupby('pseudopaz')['patient_sex'].nunique() == 1).all())

#### See which `pseudopaz` have multiple characteristics that should be unique

In [ ]:
# incons = df.groupby('pseudopaz')['patient_age'].nunique()
# incons = incons[incons > 1].sort_values(ascending=False)
# incons